# 02 — A/B Testing Framework Demo

**Goal:** Demonstrate the `framework/ab_test.py` module across three realistic business scenarios.

| # | Scenario | Test |
|---|---|---|
| 1 | Landing page redesign | Z-test + Chi-square |
| 2 | Email subject line CTR | Z-test (one-sided) |
| 3 | Pre-experiment planning | Sample size + MDE |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import json

from framework.ab_test import (
    run_ztest,
    run_chi_square,
    calculate_mde,
    calculate_sample_size,
    generate_report,
)

os.makedirs("../visuals",  exist_ok=True)
os.makedirs("../results",  exist_ok=True)
np.random.seed(42)
print("Framework imported successfully.")

---
## Scenario 1 — Landing Page Redesign

A new homepage layout is tested against the current design over 2 weeks.

| Group | Visitors | Conversions |
|---|---|---|
| Control (old page) | 8,500 | 765 |
| Treatment (new page) | 8,500 | 918 |

In [ ]:
s1_ctrl_n  = 8500;  s1_ctrl_c  = 765
s1_trt_n   = 8500;  s1_trt_c   = 918

# Z-test (two-sided — we want to know if it's different in either direction)
s1_ztest = run_ztest(
    control_conversions   = s1_ctrl_c,
    control_size          = s1_ctrl_n,
    treatment_conversions = s1_trt_c,
    treatment_size        = s1_trt_n,
    alpha                 = 0.05,
    alternative           = "two-sided",
)

print("SCENARIO 1 — Z-TEST RESULT")
print(f"  Control CVR      : {s1_ztest['control_cvr']:.2%}")
print(f"  Treatment CVR    : {s1_ztest['treatment_cvr']:.2%}")
print(f"  Absolute lift    : {s1_ztest['absolute_lift']:.2%}")
print(f"  Relative lift    : {s1_ztest['relative_lift']:.2%}")
print(f"  Z-statistic      : {s1_ztest['z_stat']}")
print(f"  P-value          : {s1_ztest['p_value']}")
print(f"  Significant      : {s1_ztest['significant']}")

In [ ]:
# Chi-square test for cross-validation
s1_chi = run_chi_square(
    control_conversions   = s1_ctrl_c,
    control_size          = s1_ctrl_n,
    treatment_conversions = s1_trt_c,
    treatment_size        = s1_trt_n,
)

print("SCENARIO 1 — CHI-SQUARE RESULT")
print(f"  Chi2 statistic   : {s1_chi['chi2_stat']}")
print(f"  P-value          : {s1_chi['p_value']}")
print(f"  Significant      : {s1_chi['significant']}")
print(f"  Cramér's V       : {s1_chi['cramers_v']}  (effect size)")

---
## Scenario 2 — Email Subject Line CTR

Marketing tests a curiosity-gap subject line vs the standard promotional subject line.

| Group | Emails Sent | Clicks |
|---|---|---|
| Control (promo subject) | 12,000 | 960 |
| Treatment (curiosity gap) | 12,000 | 1,104 |

In [ ]:
s2_ctrl_n  = 12000;  s2_ctrl_c  = 960
s2_trt_n   = 12000;  s2_trt_c   = 1104

# One-sided z-test — we only care if treatment is BETTER
s2_ztest = run_ztest(
    control_conversions   = s2_ctrl_c,
    control_size          = s2_ctrl_n,
    treatment_conversions = s2_trt_c,
    treatment_size        = s2_trt_n,
    alpha                 = 0.05,
    alternative           = "larger",   # treatment > control
)

print("SCENARIO 2 — EMAIL CTR Z-TEST (one-sided)")
print(f"  Control CTR      : {s2_ztest['control_cvr']:.2%}")
print(f"  Treatment CTR    : {s2_ztest['treatment_cvr']:.2%}")
print(f"  Absolute lift    : {s2_ztest['absolute_lift']:.2%}")
print(f"  Relative lift    : {s2_ztest['relative_lift']:.2%}")
print(f"  Z-statistic      : {s2_ztest['z_stat']}")
print(f"  P-value          : {s2_ztest['p_value']}")
print(f"  Significant      : {s2_ztest['significant']}")

---
## Scenario 3 — Pre-Experiment Planning

Before launching an experiment, the product team wants to know:
1. How many users are needed to detect a 2% absolute lift?
2. What is the MDE given the expected traffic of 3,000/group?

In [ ]:
baseline = 0.08   # 8% baseline conversion rate
target_mde = 0.02  # we want to detect a 2pp lift

# Sample size required
ss = calculate_sample_size(
    baseline_rate = baseline,
    mde           = target_mde,
    alpha         = 0.05,
    power         = 0.80,
)

print("SCENARIO 3 — SAMPLE SIZE CALCULATION")
print(f"  Baseline CVR     : {ss['baseline_rate']:.2%}")
print(f"  Target MDE       : {ss['mde']:.2%}")
print(f"  Required N/group : {ss['n_per_group']:,}")
print(f"  Total N required : {ss['total_n']:,}")

# MDE for available traffic
mde_result = calculate_mde(
    baseline_rate = baseline,
    n_per_group   = 3000,
    alpha         = 0.05,
    power         = 0.80,
)

print(f"\nWith 3,000 users/group:")
print(f"  MDE (absolute)   : {mde_result['mde_absolute']:.4f} ({mde_result['mde_absolute']:.2%})")
print(f"  MDE (relative)   : {mde_result['mde_relative']:.2%}")

In [ ]:
# Sample size sensitivity across power levels
power_levels  = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
sample_sizes  = [calculate_sample_size(baseline, target_mde, power=p)["n_per_group"] for p in power_levels]

mde_values    = [0.005, 0.01, 0.015, 0.02, 0.025, 0.03]
sample_sizes2 = [calculate_sample_size(baseline, m)["n_per_group"] for m in mde_values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(power_levels, sample_sizes, "o-", color="#4C72B0", linewidth=2, markersize=7)
for x, y in zip(power_levels, sample_sizes):
    axes[0].text(x, y + 30, f"{y:,}", ha="center", fontsize=8)
axes[0].set_xlabel("Statistical Power")
axes[0].set_ylabel("Required N per Group")
axes[0].set_title("Sample Size vs Power (MDE = 2%)", fontsize=12, fontweight="bold")
axes[0].set_xticks(power_levels)
axes[0].set_xticklabels([f"{p:.0%}" for p in power_levels])

axes[1].plot([m * 100 for m in mde_values], sample_sizes2, "o-", color="#DD8452", linewidth=2, markersize=7)
for x, y in zip(mde_values, sample_sizes2):
    axes[1].text(x * 100, y + 30, f"{y:,}", ha="center", fontsize=8)
axes[1].set_xlabel("Minimum Detectable Effect (%)")
axes[1].set_ylabel("Required N per Group")
axes[1].set_title("Sample Size vs MDE (Power = 80%)", fontsize=12, fontweight="bold")

plt.suptitle("Pre-Experiment Planning — Sample Size Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../visuals/sample_size_planning.png", bbox_inches="tight")
plt.show()
print("Saved: ../visuals/sample_size_planning.png")

## Scenario Comparison — CVR Lift Visualisation

In [ ]:
scenarios = ["S1: Landing Page", "S2: Email Subject"]
ctrl_cvrs = [s1_ztest["control_cvr"],   s2_ztest["control_cvr"]]
trt_cvrs  = [s1_ztest["treatment_cvr"], s2_ztest["treatment_cvr"]]
sig       = [s1_ztest["significant"],   s2_ztest["significant"]]

x     = np.arange(len(scenarios))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, [c*100 for c in ctrl_cvrs], width, label="Control",   color="#4C72B0", edgecolor="white")
bars2 = ax.bar(x + width/2, [t*100 for t in trt_cvrs],  width, label="Treatment", color="#DD8452", edgecolor="white")

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f"{bar.get_height():.1f}%", ha="center", fontsize=9)

for i, (sc, s) in enumerate(zip(scenarios, sig)):
    label = "✓ Significant" if s else "✗ Not Significant"
    color = "green" if s else "red"
    ax.text(i, max(trt_cvrs)*100 * 1.18, label, ha="center", color=color, fontsize=9, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.set_ylabel("Rate (%)")
ax.set_title("Scenario Comparison: Control vs Treatment CVR", fontsize=13, fontweight="bold")
ax.legend()
ax.set_ylim(0, max(trt_cvrs)*100 * 1.35)
plt.tight_layout()
plt.savefig("../visuals/scenario_comparison.png", bbox_inches="tight")
plt.show()

## Save Combined Report

In [ ]:
report_path = generate_report(
    experiment_name = "AB Testing Framework Demo",
    results = {
        "scenario_1_landing_page": {
            "ztest"      : s1_ztest,
            "chi_square" : s1_chi,
        },
        "scenario_2_email_ctr": {
            "ztest": s2_ztest,
        },
        "scenario_3_planning": {
            "sample_size": ss,
            "mde"        : mde_result,
        },
    },
    path = "../results/report.json",
    metadata = {
        "author"   : "Jayanshu Badlani",
        "framework": "ab_test.py v1.0",
    },
)

print(f"Full report saved to: {report_path}")

# Preview
with open("../results/report.json") as f:
    preview = json.load(f)
print(json.dumps(preview["results"]["scenario_1_landing_page"]["ztest"], indent=2))